Kérdés:
A kreativitás igénye csökkenti-e az automatizálás esélyét?

In [ ]:
#work environment importálása
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

In [ ]:
# Google drive csatlakozás

from google.colab import drive
drive.mount('/content/drive')

# adatok letöltése
!gdown 1UVtdO3HFR0S0gSJ7-lCo5BI55riSlJIA


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Downloading...
From: https://drive.google.com/uc?id=1UVtdO3HFR0S0gSJ7-lCo5BI55riSlJIA
To: /content/ai-impact-jobs-layoff-risk-dataset.csv
100% 1.62M/1.62M [00:00<00:00, 157MB/s]


In [ ]:
#dataframebe helyezés:
df = pd.read_csv("/content/ai-impact-jobs-layoff-risk-dataset.csv")

# **EDA Megkezdése**

In [ ]:
df.info()
#megjegyzés: nincs "null" érték, 20000 sor minden adatnál

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Age                         20000 non-null  int64 
 1   Education_Level             20000 non-null  object
 2   Years_of_Experience         20000 non-null  int64 
 3   Industry                    20000 non-null  object
 4   Job_Role                    20000 non-null  object
 5   Company_Size                20000 non-null  object
 6   Job_Level                   20000 non-null  object
 7   Routine_Task_Percentage     20000 non-null  int64 
 8   Creativity_Requirement      20000 non-null  int64 
 9   Human_Interaction_Level     20000 non-null  int64 
 10  AI_Adoption_Level           20000 non-null  object
 11  Number_of_AI_Tools_Used     20000 non-null  int64 
 12  AI_Usage_Hours_Per_Week     20000 non-null  int64 
 13  Tasks_Automated_Percentage  20000 non-null  in

In [77]:
df
#megjegyzés:

,Age,Education_Level,Years_of_Experience,Industry,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours,Layoff_Risk
0,59,Master's,6,Finance,Accountant,Medium,Entry,84,21,94,Medium,5,7,40,26,High
1,44,Master's,14,Manufacturing,Production Supervisor,Small,Entry,30,84,68,Low,2,2,14,9,Low
2,36,Bachelor's,7,Retail,Store Manager,Medium,Senior,12,86,71,Low,0,1,6,3,Low
3,27,Bachelor's,6,Finance,Auditor,Large,Entry,90,18,21,Medium,3,10,67,26,High
4,49,High School,12,Finance,Auditor,Small,Entry,49,52,72,Medium,5,13,26,19,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,56,Bachelor's,10,Retail,Sales Associate,Small,Entry,29,62,44,Low,0,4,16,0,Medium
19996,43,Bachelor's,9,Manufacturing,Production Supervisor,Medium,Entry,87,20,20,High,9,23,77,67,High
19997,28,High School,7,Telecom,Support Specialist,Medium,Entry,91,2,49,Medium,2,14,55,10,High
19998,46,Bachelor's,3,Manufacturing,Quality Engineer,Medium,Senior,76,40,89,Medium,3,13,66,5,Medium


In [78]:
df.shape

(20000, 16)

In [79]:
df.describe()

,Age,Years_of_Experience,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.00000,20000.00000,20000.000000
mean,40.361500,7.265500,51.885500,47.786700,60.055000,2.460350,6.77395,37.22895,12.621400
std,11.526333,4.720606,24.437808,26.100482,22.267801,2.097486,6.08637,19.99512,13.552647
min,21.000000,0.000000,10.000000,0.000000,20.000000,0.000000,0.00000,4.00000,0.000000
25%,30.000000,4.000000,31.000000,27.000000,41.000000,1.000000,2.00000,21.00000,4.000000
50%,40.000000,7.000000,52.000000,48.000000,62.000000,2.000000,5.00000,35.00000,8.000000
75%,50.000000,11.000000,73.000000,69.000000,79.000000,4.000000,10.00000,51.00000,18.000000
max,60.000000,32.000000,94.000000,100.000000,99.000000,10.000000,30.00000,93.00000,79.000000


# Folytonos értékek eloszlásának vizsgálata
megjegyzés hisztogrammok alatt.

In [83]:
#heti AI használat órákban
#binek száma: Max érték - Min érték +1
bs_AIuse = df['AI_Usage_Hours_Per_Week'].max() - df['AI_Usage_Hours_Per_Week'].min()+1
df['AI_Usage_Hours_Per_Week'].hist(bins=bs_AIuse)
plt.show()

erős balra húzás figyelhető meg, érdemes lehet tovább vizsgálni a megosztottság okait

In [ ]:
# életkor
#binek száma: Max érték - Min érték +1
bs_Age = df['Age'].max() - df['Age'].min()+1
df['Age'].hist(bins=bs_Age)
print(bs_Age)

40


In [ ]:
# munkatapasztalat
#binek száma: Max érték - Min érték +1
bs_yoe = df['Years_of_Experience'].max() - df['Years_of_Experience'].min()+1
df['Years_of_Experience'].hist(bins=bs_yoe)

<Axes: title={'center': 'Numerikus változók összefüggése'}>

erős balra húzás, kiugró érték magyarázható a pályakezdőkkel

In [ ]:
# rutin munkák aránya, %-os érték
#binek száma: Max érték - Min érték +1
bs_rtp = df['Routine_Task_Percentage'].max() - df['Routine_Task_Percentage'].min()+1
df['Routine_Task_Percentage'].hist(bins=bs_rtp)
print(bs_rtp)

85


egyenletes eloszlás, nincs 0% se 100%

In [ ]:
# Kreativitás igény
#binek száma: Max érték - Min érték +1
bs_cr = df['Creativity_Requirement'].max() - df['Creativity_Requirement'].min()+1
df['Creativity_Requirement'].hist(bins=bs_cr)
print(bs_cr)

101


kiugró értékek a min és max - érdemes lehet megvizsgálni iparágakra bontva, egyébként normál eloszlás figyelhető meg

In [ ]:
# emberi interakciók szükségessége %-ban
#binek száma: Max érték - Min érték +1
bs_hil = df['Human_Interaction_Level'].max() - df['Human_Interaction_Level'].min()+1
df['Human_Interaction_Level'].hist(bins=bs_hil)
print(bs_hil)

80


In [ ]:
# feladatkör automatizáltsága %
#binek száma: Max érték - Min érték +1
bs_tap = df['Tasks_Automated_Percentage'].max() - df['Tasks_Automated_Percentage'].min()+1
df['Tasks_Automated_Percentage'].hist(bins=bs_tap)
print(bs_tap)

90


In [64]:

# AI használatra biztosított képzés órákban
#binek száma: Max érték - Min érték +1
bs_ath = df['AI_Training_Hours'].max() - df['AI_Training_Hours'].min()+1
df['AI_Training_Hours'].hist(bins=bs_ath)

<Axes: title={'center': 'Numerikus változók összefüggése'}>

Classification
célváltozó = Layoff Risk

In [ ]:
# Célváltozó leválasztása

X = df.drop('Layoff_Risk', axis=1)
y = df['Layoff_Risk']  # célváltozó
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state = 11)

In [ ]:
#Pandas df-be helyezés és index-reset

X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)


In [89]:
plt.scatter(df['Creativity_Requirement'], df['Tasks_Automated_Percentage'])
plt.title('Scatter Plot of Creativity Req. vs Tasks automated %')
plt.xlabel('Creativity_Requirement')
plt.ylabel('Tasks_Automated_Percentage')
plt.show()

In [104]:
#crosstab Creativity_Requiriement vs Layoff Risk
bins = [0, 20, 40, 60, 80, 100]
labels = ['0-20', '21-40', '41-60', '61-80', '81-100']
df['Creativity_Band'] = pd.cut(df['Creativity_Requirement'], bins=bins, labels=labels)

Layoff_Risk_to_Creativity = pd.crosstab(df['Creativity_Band'], df['Layoff_Risk'], normalize='index') * 100
print(Layoff_Risk_to_Creativity[['Low', 'Medium', 'High']].round(1))

Layoff_Risk       Low  Medium  High
Creativity_Band                    
0-20              0.7    15.9  83.4
21-40             6.1    36.5  57.4
41-60            25.8    53.6  20.6
61-80            63.4    34.6   2.0
81-100           89.6    10.4   0.0


Layoff Risk a legerősebb az alacsonyabb Creativity Requirementnél.
Erős negatív korrelávió figyelhető meg a kettő között

In [106]:
# korrelációs mátrix
plt.figure(figsize=(8,5))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix,annot=True,fmt=".2f",cmap="coolwarm")
plt.title("Numerikus változók összefüggése")
plt.show()